## Import required libraries


In [1]:
import json
import os
from glob import glob

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from shapely.geometry import Point
from tqdm import tqdm

# --- Repo-relative paths ---------------------------------------------------
# Resolved from this notebook's location, so the checkout can be moved or cloned
# anywhere without editing paths here. Override with SOIL_SCENARIOS_ROOT.
import os
from pathlib import Path


def _find_repo_root(start=None):
    env = os.environ.get("SOIL_SCENARIOS_ROOT")
    if env:
        return Path(env).expanduser().resolve()
    here = Path(start or Path.cwd()).resolve()
    for cand in (here, *here.parents):
        if (cand / "simplace").is_dir() and (cand / "orchestration").is_dir():
            return cand
    raise FileNotFoundError(
        f"repo root not found from {here} (looked for simplace/ + orchestration/)"
    )


REPO_ROOT = _find_repo_root()

MAIN_DATA_DIR = os.path.join(str(REPO_ROOT), "data")
INTERIM_DATA_DIR = os.path.join(MAIN_DATA_DIR, "interim")
PROCESSED_DATA_DIR = os.path.join(MAIN_DATA_DIR, "processed")

## Read the data and filepaths


In [2]:
# Read the project file paths for main data and lai data
main_project_file_paths = sorted(
    glob(os.path.join(PROCESSED_DATA_DIR, "simplace_project_file", "main", "*.csv"))
)
lai_project_file_paths = sorted(
    glob(os.path.join(PROCESSED_DATA_DIR, "simplace_project_file", "lai", "*.csv"))
)

# Read the soil and lai coordinates
main_location_gdf = gpd.read_file(
    os.path.join(PROCESSED_DATA_DIR, "location", "main_location.gpkg")
)
lai_location_gdf = gpd.read_file(
    os.path.join(PROCESSED_DATA_DIR, "location", "lai_location.gpkg")
)

print(main_location_gdf.shape, lai_location_gdf.shape)

(3086, 5) (8848, 5)


In [3]:
# Define the crop names
crops = [
    "winter_wheat",
    "winter_rapeseed",
    "maize",
    "spring_barley",
    "sugar_beet",
    "potato",
]

## Prepare and check the soil files


In [50]:
main_soil_df = pd.read_csv(
    os.path.join(PROCESSED_DATA_DIR, "simplace_soil_file", "main_soil.csv")
)
lai_soil_df = pd.read_csv(
    os.path.join(PROCESSED_DATA_DIR, "simplace_soil_file", "lai_soil.csv"),
)

# Check if the columns are matching in both the files
main_soil_df.columns == lai_soil_df.columns

array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,

## Prepare and check fertilizer files


In [7]:
fertilizer_scenario = {
    "winter_wheat": [
        {"Event": 1, "vType": "PK", "DVS": 0.001, "Amount": 40},
        {"Event": 2, "vType": "KAS", "DVS": 0.25, "Amount": 32},
        {"Event": 3, "vType": "KAS", "DVS": 0.4, "Amount": 16},
        {"Event": 4, "vType": "KAS", "DVS": 0.9, "Amount": 16},
    ],
    "winter_rapeseed": [
        {"Event": 1, "vType": "PK", "DVS": 0.001, "Amount": 35},
        {"Event": 2, "vType": "KAS", "DVS": 0.4, "Amount": 24},
        {"Event": 3, "vType": "KAS", "DVS": 0.8, "Amount": 20},
    ],
    "maize": [
        {"Event": 1, "vType": "PK", "DVS": 0.001, "Amount": 50},
        {"Event": 2, "vType": "DAP", "DVS": 0.001, "Amount": 8},
        {"Event": 3, "vType": "KAS", "DVS": 0.4, "Amount": 24},
    ],
    "sugar_beet": [
        {"Event": 1, "vType": "PK", "DVS": 0.001, "Amount": 60},
        {"Event": 2, "vType": "KAS", "DVS": 0.01, "Amount": 30},
        {"Event": 3, "vType": "KAS", "DVS": 0.25, "Amount": 10},
    ],
    "spring_barley": [
        {"Event": 1, "vType": "PK", "DVS": 0.001, "Amount": 32},
        {"Event": 2, "vType": "KAS", "DVS": 0.01, "Amount": 31},
    ],
    "potato": [
        {"Event": 1, "vType": "Urea", "DVS": 0.001, "Amount": 8.5},
        {"Event": 1, "vType": "P", "DVS": 0.001, "Amount": 20.62},
        {"Event": 1, "vType": "K", "DVS": 0.001, "Amount": 26.49},
        {"Event": 2, "vType": "Urea", "DVS": 0.3, "Amount": 12.7},
        {"Event": 3, "vType": "Urea", "DVS": 0.5, "Amount": 17.9},
    ],
}


# Algorithm to process fertilizer scenario file
def prepare_fertilizer_file(crop):

    try:
        # Read the project files
        main_project_df = pd.read_csv(
            os.path.join(
                PROCESSED_DATA_DIR,
                "simplace_project_file",
                "main",
                f"project_{crop}.csv",
            ),
            sep=";",
        )
        main_project_df = (
            main_project_df[["vLocationID"]].drop_duplicates().reset_index(drop=True)
        )

        lai_project_df = pd.read_csv(
            os.path.join(
                PROCESSED_DATA_DIR,
                "simplace_project_file",
                "lai",
                f"project_{crop}.csv",
            ),
            sep=";",
        )
        lai_project_df = (
            lai_project_df[["vLocationID"]].drop_duplicates().reset_index(drop=True)
        )

        # Get the events dataframe
        events_df = pd.DataFrame(fertilizer_scenario[crop])

        # Define the column order
        col_order = [
            "location",
            "FertilizerScenario",
            "crop",
            "Event",
            "vType",
            "DVS",
            "Amount",
        ]

        # Process the main fertilizer file
        main_fertilizer_df = main_project_df.merge(events_df, how="cross")
        main_fertilizer_df["crop"] = crop
        main_fertilizer_df["FertilizerScenario"] = 2
        main_fertilizer_df.rename(columns={"vLocationID": "location"}, inplace=True)
        main_fertilizer_df = main_fertilizer_df[col_order]

        # process the LAI fertilizer file
        lai_fertilizer_df = lai_project_df.merge(events_df, how="cross")
        lai_fertilizer_df["crop"] = crop
        lai_fertilizer_df["FertilizerScenario"] = 2
        lai_fertilizer_df.rename(columns={"vLocationID": "location"}, inplace=True)
        lai_fertilizer_df = lai_fertilizer_df[col_order]

        # Save the data
        main_fertilizer_path = os.path.join(
            PROCESSED_DATA_DIR,
            "simplace_fertilizer_file",
            "main",
            f"fertilizer_{crop}.csv",
        )
        lai_fertilizer_path = os.path.join(
            PROCESSED_DATA_DIR,
            "simplace_fertilizer_file",
            "lai",
            f"fertilizer_{crop}.csv",
        )

        main_fertilizer_df.to_csv(main_fertilizer_path, index=False)
        lai_fertilizer_df.to_csv(lai_fertilizer_path, index=False)

        print("*" * 20 + crop + "*" * 20)
        print(f"Main Fertilizer file saved at path: {main_fertilizer_path}")
        print(f"LAI Fertilizer file saved at path: {lai_fertilizer_path}")
        print("\n")

    except:
        print("*" * 20 + crop + "*" * 20)
        print(f"Input file not found for crop:{crop}")
        print("\n")


# # Run the algorithm
# for crop in crops:
#     prepare_fertilizer_file(crop)

## Prepare and check location files


In [8]:
# Algorithm to process location files
def prepare_location_file(crop):

    lai_project_df = pd.read_csv(
        os.path.join(
            PROCESSED_DATA_DIR,
            "simplace_project_file",
            "lai",
            f"project_{crop}.csv",
        ),
        sep=";",
    )

    lai_location_gdf = gpd.read_file(
        os.path.join(PROCESSED_DATA_DIR, "location", "lai_location.gpkg")
    )
    lai_location_gdf = lai_location_gdf[
        lai_location_gdf["PointID"].isin(lai_project_df["vLocationID"].unique())
    ]
    lai_location_gdf["location"] = lai_location_gdf[["PointID"]]
    lai_location_gdf["Latitude"] = lai_location_gdf.geometry.y
    lai_location_gdf = lai_location_gdf[["location", "Latitude"]]
    lai_location_gdf["SunInclination"] = -4
    lai_location_gdf["Altitude"] = 10

    lai_location_path = os.path.join(
        PROCESSED_DATA_DIR,
        "simplace_location_file",
        "lai",
        f"location_{crop}.csv",
    )

    lai_location_gdf.to_csv(lai_location_path, index=False)
    print("*" * 20 + crop + "*" * 20)
    print(f"LAI location file saved at path: {lai_location_path}")

    return lai_location_gdf


# # Run the algorithm
# for crop in crops:
#     prepare_location_file(crop)